**installing dependencies**

In [ ]:
!pip install torch transformers datasets evaluate sacrebleu sentencepiece ctranslate2 accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.2 MB/s eta 0:00:00


In [ ]:
!python train_translator.py

Loading dataset splits...
Loaded 47500 training pairs and 2500 validation pairs.
Loading base model and tokenizer: facebook/nllb-200-distilled-600M...
config.json: 100% 846/846 [00:00<00:00, 3.33MB/s]
tokenizer_config.json: 100% 564/564 [00:00<00:00, 1.91MB/s]

sentencepiece.bpe.model: downloading bytes:  59% 2.87M/4.85M [00:01<00:00, 2.43MB/s]
sentencepiece.bpe.model: downloading bytes: 100% 3.48M/3.48M [00:01<00:00, 2.93MB/s,  338kB/s  ]
sentencepiece.bpe.model: reconstructing file: 100% 4.85M/4.85M [00:01<00:00, 4.08MB/s,  471kB/s  ]

tokenizer.json: downloading bytes:  45% 7.83M/17.3M [00:01<00:01, 6.99MB/s]
tokenizer.json: downloading bytes: 100% 7.83M/7.83M [00:01<00:00, 6.93MB/s,  760kB/s  ]
tokenizer.json: reconstructing file: 100% 17.3M/17.3M [00:01<00:00, 15.4MB/s, 1.68MB/s  ]
special_tokens_map.json: 100% 3.55k/3.55k [00:00<00:00, 11.7MB/s]

pytorch_model.bin: downloading bytes:   8% 190M/2.46G [00:01<00:11, 203MB/s, 15.0MB/s  ]
pytorch_model.bin: downloading bytes:   9% 231

In [ ]:
!sed -i 's/evaluation_strategy/eval_strategy/g' train_translator.py
!python train_translator.py

Loading dataset splits...
Loaded 47500 training pairs and 2500 validation pairs.
Loading base model and tokenizer: facebook/nllb-200-distilled-600M...
Loading weights: 100% 512/512 [00:00<00:00, 13699.88it/s]
Tokenizing dataset...
Map: 100% 47500/47500 [00:05<00:00, 8954.73 examples/s]
Map: 100% 2500/2500 [00:00<00:00, 8768.85 examples/s]
Traceback (most recent call last):
  File "/content/train_translator.py", line 77, in <module>
    trainer = Seq2SeqTrainer(
        model=model,
    ...<5 lines>...
        compute_metrics=compute_metrics,
    )
TypeError: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'


In [ ]:
!sed -i '/tokenizer=tokenizer/d' train_translator.py
!sed -i 's/evaluation_strategy/eval_strategy/g' train_translator.py
!python train_translator.py

Loading dataset splits...
Loaded 47500 training pairs and 2500 validation pairs.
Loading base model and tokenizer: facebook/nllb-200-distilled-600M...
Loading weights: 100% 512/512 [00:00<00:00, 19466.13it/s]
Tokenizing dataset...
Map: 100% 47500/47500 [00:06<00:00, 7634.12 examples/s] 
Map: 100% 2500/2500 [00:00<00:00, 9494.31 examples/s]
Starting model fine-tuning...
{'loss': '1.271', 'grad_norm': '2.817', 'learning_rate': '2.983e-05', 'epoch': '0.01684'}
{'loss': '1.222', 'grad_norm': '2.662', 'learning_rate': '2.966e-05', 'epoch': '0.03368'}
{'loss': '1.216', 'grad_norm': '3.406', 'learning_rate': '2.95e-05', 'epoch': '0.05052'}
{'loss': '1.209', 'grad_norm': '3.216', 'learning_rate': '2.933e-05', 'epoch': '0.06736'}
{'loss': '1.161', 'grad_norm': '3.244', 'learning_rate': '2.916e-05', 'epoch': '0.0842'}
{'loss': '1.163', 'grad_norm': '2.29', 'learning_rate': '2.899e-05', 'epoch': '0.101'}
{'loss': '1.167', 'grad_norm': '2.7', 'learning_rate': '2.882e-05', 'epoch': '0.1179'}
{'loss

In [2]:
# 1. Install missing packages in Colab
!pip install -q evaluate sacrebleu ctranslate2 accelerate datasets transformers sentencepiece pandas pyarrow

import json
import os
import subprocess
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

TRAIN_PATH = "train.json"
VAL_PATH = "val.json"
OUTPUT_DIR = "./output_checkpoints"
FINAL_MODEL_DIR = "./final_manhwa_translator"
CT2_MODEL_DIR = "./nmt_webtoon_ct2"

# 2. Auto-create train.json and val.json if Colab reset overnight
if not os.path.exists(TRAIN_PATH) or not os.path.exists(VAL_PATH):
    print("Dataset files not found in Colab workspace. Downloading from Hugging Face...")
    PARQUET_URL = "https://huggingface.co/api/datasets/traintogpb/aihub-koen-translation-integrated-large-10m/parquet/default/train/0.parquet"
    df = pd.read_parquet(PARQUET_URL, columns=["ko", "en"])
    samples = []
    for _, row in df.iterrows():
        ko = str(row["ko"]).strip()
        en = str(row["en"]).strip()
        if 5 <= len(ko) <= 120 and 5 <= len(en) <= 150:
            samples.append({"ko": ko, "en": en})
        if len(samples) >= 50000:
            break
    split_idx = int(len(samples) * 0.95)
    with open(TRAIN_PATH, "w", encoding="utf-8") as f:
        json.dump(samples[:split_idx], f, ensure_ascii=False, indent=2)
    with open(VAL_PATH, "w", encoding="utf-8") as f:
        json.dump(samples[split_idx:], f, ensure_ascii=False, indent=2)
    print("Created train.json and val.json!")

# 3. Load Dataset
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "kor_Hang"
TGT_LANG = "eng_Latn"

print("Loading dataset splits...")
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)[:500]  # 500 val samples for fast evaluation

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Loaded {len(train_dataset)} training pairs and {len(val_dataset)} validation pairs.")

print(f"Loading base model and tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = [ex for ex in batch["ko"]]
    targets = [ex for ex in batch["en"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset...")
tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=["ko", "en"])
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=["ko", "en"])

# 4. Configure Training Arguments (Optimized for Colab T4 GPU)
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=1500,
    save_steps=1500,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none"
)

metric = evaluate.load("sacrebleu")
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    # Replace -100 with pad_token_id to prevent Rust tokenizer OverflowError
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": round(result["score"], 2)}

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

print("Starting model fine-tuning...")
resume_cp = True if os.path.exists(OUTPUT_DIR) and any(d.startswith("checkpoint-") for d in os.listdir(OUTPUT_DIR)) else None
trainer.train(resume_from_checkpoint=resume_cp)

print(f"Saving final model to {FINAL_MODEL_DIR}...")
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# 5. Convert to CTranslate2 INT8 and Zip for Download
print("Converting model to CTranslate2 INT8 format...")
subprocess.run([
    "ct2-transformers-converter",
    "--model", FINAL_MODEL_DIR,
    "--output_dir", CT2_MODEL_DIR,
    "--quantization", "int8",
    "--force"
], check=True)

# Copy tokenizer files into CT2 folder so local inference has everything it needs
subprocess.run(f"cp {FINAL_MODEL_DIR}/sentencepiece.bpe.model {CT2_MODEL_DIR}/", shell=True)
subprocess.run(f"cp {FINAL_MODEL_DIR}/tokenizer.json {CT2_MODEL_DIR}/", shell=True)
subprocess.run(f"cp {FINAL_MODEL_DIR}/special_tokens_map.json {CT2_MODEL_DIR}/", shell=True)
subprocess.run(f"cp {FINAL_MODEL_DIR}/tokenizer_config.json {CT2_MODEL_DIR}/", shell=True)

subprocess.run(["zip", "-r", "nmt_webtoon_ct2.zip", CT2_MODEL_DIR], check=True)
print("ALL DONE! Download 'nmt_webtoon_ct2.zip' from the Colab Files sidebar on the left.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.4 MB/s eta 0:00:00
Dataset files not found in Colab workspace. Downloading from Hugging Face...
Created train.json and val.json!
Loading dataset splits...
Loaded 47500 training pairs and 500 validation pairs.
Loading base model and tokenizer: facebook/nllb-200-distilled-600M...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Map:   0%|          | 0/47500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Starting model fine-tuning...


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 720.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 245.81 MiB is free. Including non-PyTorch memory, this process has 14.32 GiB memory in use. Of the allocated memory 12.53 GiB is allocated by PyTorch, and 1.65 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [3]:
import gc
import json
import os
import subprocess
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

# 1. Free any lingering GPU memory from the previous run
for var in ["model", "trainer"]:
    if var in globals():
        del globals()[var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TRAIN_PATH = "train.json"
VAL_PATH = "val.json"
OUTPUT_DIR = "./output_checkpoints"
FINAL_MODEL_DIR = "./final_manhwa_translator"
CT2_MODEL_DIR = "./nmt_webtoon_ct2"

# 2. Load Dataset
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "kor_Hang"
TGT_LANG = "eng_Latn"

print("Loading dataset splits...")
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)[:500]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Loaded {len(train_dataset)} training pairs and {len(val_dataset)} validation pairs.")

print(f"Loading base model and tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = [ex for ex in batch["ko"]]
    targets = [ex for ex in batch["en"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset...")
tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=["ko", "en"])
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=["ko", "en"])

# 3. Memory-Safe Training Arguments (~5.2 GB VRAM on T4 GPU, ~29 mins total)
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=1500,
    save_steps=1500,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    optim="adafactor",
    gradient_checkpointing=True,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none"
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-finetuned",
    per_device_train_batch_size=4,      # <--- Change to 4
    per_device_eval_batch_size=4,       # <--- Change to 4
    gradient_accumulation_steps=8,      # <--- Add this
    fp16=True,                          # <--- Add this (cuts memory in half)
    gradient_checkpointing=True,        # <--- Add this (saves ~5GB VRAM)
    optim="adafactor",                  # <--- Add this (saves ~4GB VRAM)
    # ... keep the rest of your arguments (learning_rate, epochs, etc.) the same
)

metric = evaluate.load("sacrebleu")
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": round(result["score"], 2)}

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)



print("Starting model fine-tuning...")
resume_cp = True if os.path.exists(OUTPUT_DIR) and any(d.startswith("checkpoint-") for d in os.listdir(OUTPUT_DIR)) else None
trainer.train(resume_from_checkpoint=resume_cp)

print(f"Saving final model to {FINAL_MODEL_DIR}...")
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# 4. Convert to CTranslate2 INT8 and Zip for Download
print("Converting model to CTranslate2 INT8 format...")
subprocess.run([
    "ct2-transformers-converter",
    "--model", FINAL_MODEL_DIR,
    "--output_dir", CT2_MODEL_DIR,
    "--quantization", "int8",
    "--force"
], check=True)

subprocess.run(f"cp {FINAL_MODEL_DIR}/sentencepiece.bpe.model {CT2_MODEL_DIR}/", shell=True)
subprocess.run(f"cp {FINAL_MODEL_DIR}/tokenizer.json {CT2_MODEL_DIR}/", shell=True)
subprocess.run(f"cp {FINAL_MODEL_DIR}/special_tokens_map.json {CT2_MODEL_DIR}/", shell=True)
subprocess.run(f"cp {FINAL_MODEL_DIR}/tokenizer_config.json {CT2_MODEL_DIR}/", shell=True)

subprocess.run(["zip", "-r", "nmt_webtoon_ct2.zip", CT2_MODEL_DIR], check=True)
print("ALL DONE! Download 'nmt_webtoon_ct2.zip' from the Colab Files sidebar on the left.")

Loading dataset splits...
Loaded 47500 training pairs and 500 validation pairs.
Loading base model and tokenizer: facebook/nllb-200-distilled-600M...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Tokenizing dataset...


Map:   0%|          | 0/47500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 245.81 MiB is free. Including non-PyTorch memory, this process has 14.32 GiB memory in use. Of the allocated memory 12.53 GiB is allocated by PyTorch, and 1.65 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

new

In [ ]:
import gc, json, os, subprocess, torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TRAIN_PATH, VAL_PATH = "train.json", "val.json"
OUTPUT_DIR = "./output_checkpoints"
FINAL_MODEL_DIR = "./final_manhwa_translator"
CT2_MODEL_DIR = "./nmt_webtoon_ct2"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_dataset = Dataset.from_list(json.load(f))
with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_dataset = Dataset.from_list(json.load(f)[:500])

MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="kor_Hang", tgt_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.config.use_cache = False

def preprocess(batch):
    model_inputs = tokenizer(batch["ko"], max_length=128, truncation=True)
    labels = tokenizer(text_target=batch["en"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=["ko", "en"])
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=["ko", "en"])

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=1500,
    save_steps=1500,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    optim="adafactor",
    gradient_checkpointing=True,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none"
)

metric = evaluate.load("sacrebleu")
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    decoded_labels = [[l.strip()] for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    res = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": round(res["score"], 2)}

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

trainer.train()

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

subprocess.run(["ct2-transformers-converter", "--model", FINAL_MODEL_DIR, "--output_dir", CT2_MODEL_DIR, "--quantization", "int8", "--force"], check=True)
for fn in ["sentencepiece.bpe.model", "tokenizer.json", "special_tokens_map.json", "tokenizer_config.json"]:
    subprocess.run(f"cp {FINAL_MODEL_DIR}/{fn} {CT2_MODEL_DIR}/", shell=True)
subprocess.run(["zip", "-r", "nmt_webtoon_ct2.zip", CT2_MODEL_DIR], check=True)
print("ALL DONE! Download nmt_webtoon_ct2.zip from the Files panel.")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Map:   0%|          | 0/47500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Bleu
1500,4.269209,1.052504,34.200000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Bleu
1500,4.269209,1.052504,34.200000


In [4]:
import os, subprocess

print("Checking saved files on Colab disk...")
print("Current directory contents:", os.listdir("."))

if os.path.exists("nmt_webtoon_ct2.zip"):
    print("✅ nmt_webtoon_ct2.zip is ALREADY created and ready to download!")
else:
    # Find the latest saved model directory
    model_dir = None
    if os.path.exists("./final_manhwa_translator") and os.listdir("./final_manhwa_translator"):
        model_dir = "./final_manhwa_translator"
    elif os.path.exists("./output_checkpoints"):
        cps = sorted(
            [d for d in os.listdir("./output_checkpoints") if d.startswith("checkpoint-")],
            key=lambda x: int(x.split("-")[1])
        )
        if cps:
            model_dir = os.path.join("./output_checkpoints", cps[-1])

    if model_dir:
        print(f"✅ Found trained model at: {model_dir}! Converting to CTranslate2...")
        !pip install -q ctranslate2 transformers sentencepiece
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="kor_Hang", tgt_lang="eng_Latn")
        tokenizer.save_pretrained(model_dir)

        CT2_MODEL_DIR = "./nmt_webtoon_ct2"
        subprocess.run(["ct2-transformers-converter", "--model", model_dir, "--output_dir", CT2_MODEL_DIR, "--quantization", "int8", "--force"], check=True)
        for fn in ["sentencepiece.bpe.model", "tokenizer.json", "special_tokens_map.json", "tokenizer_config.json"]:
            src = os.path.join(model_dir, fn)
            if os.path.exists(src):
                subprocess.run(f"cp {src} {CT2_MODEL_DIR}/", shell=True)
        subprocess.run(["zip", "-r", "nmt_webtoon_ct2.zip", CT2_MODEL_DIR], check=True)
        print("✅ ALL DONE! Download nmt_webtoon_ct2.zip from the Files sidebar on the left!")
    else:
        print("⚠️ Colab wiped the temporary disk upon disconnect.")

Checking saved files on Colab disk...
Current directory contents: ['.config', 'final_manhwa_translator', 'nmt_webtoon_ct2', 'nmt_webtoon_ct2.zip', 'output_checkpoints', 'val.json', 'train.json', 'sample_data']
✅ nmt_webtoon_ct2.zip is ALREADY created and ready to download!


In [5]:
from google.colab import drive
drive.mount('/content/drive')
!cp nmt_webtoon_ct2.zip /content/drive/MyDrive/
print("✅ Copied to your Google Drive! Open drive.google.com and download nmt_webtoon_ct2.zip")

Mounted at /content/drive
✅ Copied to your Google Drive! Open drive.google.com and download nmt_webtoon_ct2.zip
